In [1]:
import sys,os
notebook_dir = os.getcwd()  # Gets current working directory
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)
sys.path.append(parent_dir + "/python_code")

In [2]:
from implementation.Cloth import Cloth
from implementation.utils import createRectangularMesh
from implementation.Gripper_dq import (
    SimulateGripper,
    quat_from_axis_angle,
    dq_from_rt,
)
import numpy as np


In [3]:
na = 20
nb = 30
np.random.seed(1)

In [4]:
X, T = createRectangularMesh(a=0.5, b=0.8, na=na, nb=nb, h=0.2)
X[:, 2] += 0.7
X += 0.0001 * np.random.randn(X.shape[0], 3)

In [5]:
cloth = Cloth(X, T)
dt = cloth.estimateTimeStep(L=0.8)
cloth.setSimulatorParameters(dt=dt)

Based on your mesh, your best dt is: 0.0022705402708232698


In [6]:
grip = SimulateGripper(cloth)

tf = int(6 / dt)
grasp_nodes = [0, 1]

In [7]:
# Initial pose at selected nodes
q0 = np.array([1.0, 0.0, 0.0, 0.0], dtype=float)
p0 = cloth.positions[grasp_nodes].mean(axis=0)

In [8]:
q0, p0

(array([1., 0., 0., 0.]), array([-0.23681454, -0.39998732,  0.88860505]))

In [9]:
dq0 = dq_from_rt(q0, p0)
grip.set_pose(dq=dq0)
grip.select_nodes(grasp_nodes)

In [10]:
t_start = 2.0
t_rot = 4.0
rot_angle = 0.5 * np.pi

axis_dir = np.array([0.0, 0.0, 1.0], dtype=float)
axis_dir /= np.linalg.norm(axis_dir)
axis_center = np.array([0.0, 0.0, 0.0], dtype=float)

axis_dir, axis_center

(array([0., 0., 1.]), array([0., 0., 0.]))

In [11]:
cloth.positions[20]

array([-0.25007544, -0.37228851,  0.88755129])

In [12]:
# cloth.plotMesh()

In [13]:
## z-direction of the gripper, normal to nodes
P = cloth.positions[grasp_nodes]
# P = np.asarray(P, dtype=float)
c = P.mean(axis=0)
Q = P - c
_, _, Vt = np.linalg.svd(Q, full_matrices=False)
n = Vt[-1]
z_ax = n/np.linalg.norm(n)
z_ax

array([-0.08873845,  0.06170173,  0.99414203])

In [14]:
## x-direction of the gripper
d = Vt[0]
d = d - np.dot(d, z_ax) * z_ax
x_ax = d/np.linalg.norm(d)
x_ax


array([0.99605495, 0.00564898, 0.0885586 ])

In [15]:
y = np.cross(z_ax, x_ax)
y_ax = y/np.linalg.norm(y)
x = np.cross(y_ax, z_ax)
x_ax = x/np.linalg.norm(x)
x_ax, y_ax

(array([0.99605495, 0.00564898, 0.0885586 ]),
 array([-1.51669608e-04,  9.98078647e-01, -6.19595945e-02]))

In [24]:
import numpy as np
import polyscope as ps

ps.init()
ps.remove_all_structures()


# 4 vertices
nodes = np.array([
    [0.0, 0.0, 0.0],   # 0
    [1.0, 0.0, 0.0],   # 1
    [0.5, 1.0, 0.0],   # 2
    [1.5, 1.0, 0.0],   # 3
])

# unique edges
edges = np.array([
    [0, 1],
    [1, 2],
    [2, 0],   # triangle 1

    [1, 3],
    [3, 2],
    [2, 1]   # triangle 2
    # shared edge (1,2) already included
])

ps.register_curve_network(
    "two_triangles",
    nodes,
    edges,
    radius=0.01,
    color=[0.0, 1.0, 1.0]
)

ps.show()

In [22]:
ps.remove_all_structures()

# 4 vertices
nodes = np.array([
    [0.0, 0.0, 0.0],   # 0
    [1.0, 0.0, 0.0],   # 1
    [0.5, 1.0, 0.0],   # 2
    [1.5, 1.0, 0.0],   # 3
])
faces = np.array([
    [0, 1, 2],
    [1, 3, 2]
])

ps.register_surface_mesh("mesh", nodes, faces)

ps.show()